# CDT v2: DNA + RNA Only (Protein-Free)

## Overview
- **Goal**: Remove Protein branch completely, use only DNA + RNA
- **Reason**: Most experimenters don't have Proteomics data
- **Base**: CDT_v3_5_RawExpression_Training_v2.ipynb (r = 0.5034)

## Architecture
```
DNA [896, 3072] → Projector → Self-Attn(2層)
                      ↓ Cross-Attn
RNA [2360] → RawExpressionEncoder → Self-Attn(1層)
                      ↓
                    VCE (2モダリティプール)
                      ↓
                    Task Layer → [n_genes]
```

## Changes from 3-modality version
- ❌ Removed: `protein_projector`
- ❌ Removed: `protein_self_attn_layers`
- ❌ Removed: `rna_to_protein` Cross-Attention
- ❌ Removed: VCE `protein_query` and related pooling
- ❌ Removed: ProteomeLM file loading
- ✅ VCE fusion: `d_model * 3` → `d_model * 2`

## Expected Performance
| Version | DNA | RNA | Protein | r |
|---------|-----|-----|---------|---|
| v3.3.1 | Enformer | scGPT | ProteomeLM | 0.50 |
| Raw v2 | Enformer | Raw | ProteomeLM | 0.5034 |
| **DNA+RNA** | Enformer | Raw | **None** | ??? |

## 1. Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Check GPU & PyTorch
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Check RAM
import psutil
ram_gb = psutil.virtual_memory().total / 1e9
print(f"\nSystem RAM: {ram_gb:.1f} GB")

In [ ]:
!pip install h5py tqdm scipy -q
print("Done!")

In [ ]:
import os
import gzip
import h5py
import json
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from tqdm import tqdm
from datetime import datetime
import urllib.request
from scipy.io import mmread

print("All imports successful!")

## 2. Paths Configuration (Protein Removed)

In [ ]:
# Google Drive paths
DRIVE_BASE = Path("/content/drive/MyDrive/cdt_data")
OUTPUT_BASE = Path("/content/drive/MyDrive/cdt_outputs/v2_dna_rna_only")
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

# DNA embeddings
DNA_PATH = DRIVE_BASE / "pilot_full_v2.h5"

# NO PROTEIN! (Removed)
# PROTEIN_PATH = DRIVE_BASE / "human_proteomelm_embeddings_aligned.h5"  # NOT NEEDED

# scRNA-seq data (for raw expression)
SCRNA_DIR = DRIVE_BASE / "gasperini_scrna"
EXPRS_PATH = SCRNA_DIR / "GSE120861_at_scale_screen.exprs.mtx"
GENES_PATH = SCRNA_DIR / "GSE120861_at_scale_screen.genes.txt.gz"
CELLS_PATH = SCRNA_DIR / "GSE120861_at_scale_screen.cells.txt.gz"

# Cell-enhancer mapping
MAPPING_PATH = DRIVE_BASE / "cell_enhancer_mapping.json"

# Target gene list (v3.3.1 2360 genes)
ALIGNED_GENES_PATH = DRIVE_BASE / "k562_gene_embeddings_aligned.h5"

# Training data
TRAIN_PATH = DRIVE_BASE / "training/gasperini_train.h5"
VAL_PATH = DRIVE_BASE / "training/gasperini_val.h5"

# Check files
print("Checking files...")
for name, path in [
    ("DNA embeddings", DNA_PATH),
    ("Expression matrix", EXPRS_PATH),
    ("Cell-enhancer mapping", MAPPING_PATH),
    ("Target genes (2360)", ALIGNED_GENES_PATH),
    ("Train data", TRAIN_PATH),
    ("Val data", VAL_PATH)
]:
    status = "OK" if path.exists() else "NOT FOUND"
    print(f"  [{status}] {name}")

print("\n** CDT v2: DNA + RNA Only (NO PROTEIN) **")

## 3. Load scRNA-seq Data and Build Per-Enhancer Expression

In [ ]:
# Download Ensembl GTF for ENSG -> Symbol mapping
print("Downloading Ensembl gene annotation...")

gtf_url = "https://ftp.ensembl.org/pub/release-109/gtf/homo_sapiens/Homo_sapiens.GRCh38.109.gtf.gz"
gtf_path = "/content/Homo_sapiens.GRCh38.109.gtf.gz"

if not os.path.exists(gtf_path):
    urllib.request.urlretrieve(gtf_url, gtf_path)
    print("Downloaded GTF file")
else:
    print("Using cached GTF file")

# Parse GTF for ENSG -> Symbol mapping
print("Parsing GTF for gene names...")
ENSG_TO_SYMBOL = {}

with gzip.open(gtf_path, 'rt') as f:
    for line in f:
        if line.startswith('#'):
            continue
        fields = line.strip().split('\t')
        if len(fields) < 9 or fields[2] != 'gene':
            continue
        
        attrs = fields[8]
        gene_id = gene_name = None
        
        for attr in attrs.split(';'):
            attr = attr.strip()
            if attr.startswith('gene_id'):
                gene_id = attr.split('"')[1].split('.')[0]
            elif attr.startswith('gene_name'):
                gene_name = attr.split('"')[1]
        
        if gene_id and gene_name:
            ENSG_TO_SYMBOL[gene_id] = gene_name

print(f"Mapped {len(ENSG_TO_SYMBOL)} genes from GTF")

In [ ]:
# Load target gene list (v3.3.1 - 2360 genes)
print("Loading target genes (2360)...")
with h5py.File(ALIGNED_GENES_PATH, 'r') as f:
    TARGET_GENES = [g.decode() if isinstance(g, bytes) else g for g in f['gene_names'][:]]

print(f"Target genes: {len(TARGET_GENES)}")
print(f"First 10: {TARGET_GENES[:10]}")

# Build symbol -> index mapping
SYMBOL_TO_IDX = {name: i for i, name in enumerate(TARGET_GENES)}

In [ ]:
# Load expression matrix
print("Loading expression matrix (this may take a few minutes)...")

expr_matrix = mmread(EXPRS_PATH)
print(f"Expression matrix shape: {expr_matrix.shape}")

# Transpose: genes x cells -> cells x genes, then to CSR for row slicing
expr_csr = expr_matrix.T.tocsr()
print(f"Transposed to CSR: {expr_csr.shape} (cells x genes)")

# Load gene names (ENSG IDs)
with gzip.open(GENES_PATH, 'rt') as f:
    expr_genes = [line.strip() for line in f]
print(f"Expression genes: {len(expr_genes)}")

# Load cell barcodes
with gzip.open(CELLS_PATH, 'rt') as f:
    expr_cells = [line.strip() for line in f]
print(f"Cells: {len(expr_cells)}")

# Build barcode -> expression matrix index mapping
expr_cell_to_idx = {bc: i for i, bc in enumerate(expr_cells)}

# Map expression genes (ENSG) to symbols
expr_gene_symbols = []
for ensg in expr_genes:
    ensg_base = ensg.split('.')[0]
    symbol = ENSG_TO_SYMBOL.get(ensg_base, None)
    expr_gene_symbols.append(symbol)

# Build symbol -> expression gene index mapping
symbol_to_expr_idx = {}
for i, symbol in enumerate(expr_gene_symbols):
    if symbol is not None and symbol not in symbol_to_expr_idx:
        symbol_to_expr_idx[symbol] = i

# Map target genes to expression matrix indices
target_to_expr_idx = {}
for i, gene in enumerate(TARGET_GENES):
    if gene in symbol_to_expr_idx:
        target_to_expr_idx[i] = symbol_to_expr_idx[gene]

print(f"Target genes in expression matrix: {len(target_to_expr_idx)} / {len(TARGET_GENES)}")

In [ ]:
# Load cell-enhancer mapping
print("Loading cell-enhancer mapping...")
with open(MAPPING_PATH, 'r') as f:
    mapping_data = json.load(f)

enhancer_to_cells = mapping_data['enhancer_to_cells']
cell_barcodes = mapping_data['cell_barcodes']

print(f"Enhancers with cells: {len(enhancer_to_cells)}")
print(f"Total cells in mapping: {len(cell_barcodes)}")

In [ ]:
# Load training data coordinates to get list of enhancers
print("Loading training data coordinates...")

with h5py.File(TRAIN_PATH, 'r') as f:
    train_chr = [c.decode() if isinstance(c, bytes) else c for c in f['enhancer_chr'][:]]
    train_start = f['enhancer_start'][:]
    train_end = f['enhancer_end'][:]

# Build unique coordinate list
unique_coords = []
seen = set()
for i in range(len(train_chr)):
    center = (train_start[i] + train_end[i]) // 2
    coord_key = f"{train_chr[i]}:{center}"
    if coord_key not in seen:
        unique_coords.append((train_chr[i], center))
        seen.add(coord_key)

print(f"Unique enhancers: {len(unique_coords)}")

In [ ]:
import random
random.seed(42)
np.random.seed(42)

# Configuration
MAX_CELLS_PER_ENHANCER = 100  # Number of cells to sample per enhancer

def get_per_enhancer_expression(coord, max_cells=100):
    """
    Get raw expression from perturbed cells for each enhancer and average.
    
    Returns:
        expression: [n_target_genes] normalized log expression
    """
    coord_key = f"{coord[0]}:{coord[1]}"
    
    if coord_key not in enhancer_to_cells:
        return np.zeros(len(TARGET_GENES), dtype=np.float32)
    
    # Get cell indices for this enhancer
    mapping_cell_indices = enhancer_to_cells[coord_key]
    
    # Convert to expression matrix indices
    expr_cell_indices = []
    for map_idx in mapping_cell_indices:
        barcode = cell_barcodes[map_idx]
        if barcode in expr_cell_to_idx:
            expr_cell_indices.append(expr_cell_to_idx[barcode])
    
    if not expr_cell_indices:
        return np.zeros(len(TARGET_GENES), dtype=np.float32)
    
    # Sample cells if too many
    if len(expr_cell_indices) > max_cells:
        expr_cell_indices = random.sample(expr_cell_indices, max_cells)
    
    # Get expression for all sampled cells and average
    cell_expressions = expr_csr[expr_cell_indices, :].toarray()  # [n_cells, n_expr_genes]
    avg_expression = cell_expressions.mean(axis=0)  # [n_expr_genes]
    
    # Map to target genes
    target_expression = np.zeros(len(TARGET_GENES), dtype=np.float32)
    for target_idx, expr_idx in target_to_expr_idx.items():
        target_expression[target_idx] = avg_expression[expr_idx]
    
    # Normalize: CPM + log1p
    total = target_expression.sum()
    if total > 0:
        target_expression = target_expression * 10000 / total
    target_expression = np.log1p(target_expression)
    
    return target_expression

# Test
test_expr = get_per_enhancer_expression(unique_coords[0])
print(f"Test expression shape: {test_expr.shape}")
print(f"  Non-zero: {(test_expr > 0).sum()}")
print(f"  Mean: {test_expr.mean():.4f}")
print(f"  Max: {test_expr.max():.4f}")

In [ ]:
# Pre-compute per-enhancer expressions for all enhancers
print(f"Pre-computing per-enhancer expressions ({len(unique_coords)} enhancers)...")
print(f"Max cells per enhancer: {MAX_CELLS_PER_ENHANCER}")

PERENHANCER_EXPRESSION = {}  # (chrom, center) -> expression

for coord in tqdm(unique_coords, desc="Computing expressions"):
    expr = get_per_enhancer_expression(coord, max_cells=MAX_CELLS_PER_ENHANCER)
    PERENHANCER_EXPRESSION[coord] = expr

print(f"\nComputed {len(PERENHANCER_EXPRESSION)} per-enhancer expressions")

# Convert to numpy array for efficient access
coord_to_idx = {coord: i for i, coord in enumerate(unique_coords)}
SHARED_RNA_EXPR = np.stack([PERENHANCER_EXPRESSION[coord] for coord in unique_coords])
print(f"Stacked expression shape: {SHARED_RNA_EXPR.shape}")

## 4. Model Definition (DNA + RNA Only)

In [ ]:
from dataclasses import dataclass
from typing import Optional


@dataclass
class CDTDNARNAConfig:
    """CDT Config for DNA + RNA only (no Protein)"""
    dna_dim: int = 3072
    dna_seq_len: int = 896
    # protein_dim: int = 768  # REMOVED
    n_genes: int = 2360
    hidden_dim: int = 768
    nhead: int = 8
    dropout: float = 0.3
    dna_self_attn_layers: int = 2
    rna_self_attn_layers: int = 1
    # protein_self_attn_layers: int = 1  # REMOVED


class RawExpressionEncoder(nn.Module):
    """
    Raw expression -> hidden_dim embeddings
    
    Each gene gets:
    - A learned gene identity embedding
    - A projection of its expression value
    """
    def __init__(self, n_genes: int, hidden_dim: int, dropout: float = 0.1):
        super().__init__()
        self.n_genes = n_genes
        self.hidden_dim = hidden_dim
        
        # Gene identity embedding (learned)
        self.gene_embedding = nn.Embedding(n_genes, hidden_dim)
        
        # Expression value projection
        self.expr_projector = nn.Sequential(
            nn.Linear(1, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        # Combine gene identity and expression
        self.combine = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout)
        )
        
    def forward(self, expression: torch.Tensor) -> torch.Tensor:
        """
        Args:
            expression: [batch, n_genes] raw expression values
        
        Returns:
            embeddings: [batch, n_genes, hidden_dim]
        """
        batch_size = expression.size(0)
        device = expression.device
        
        # Gene identity embeddings (same for all samples)
        gene_ids = torch.arange(self.n_genes, device=device)
        gene_emb = self.gene_embedding(gene_ids)  # [n_genes, hidden_dim]
        gene_emb = gene_emb.unsqueeze(0).expand(batch_size, -1, -1)  # [batch, n_genes, hidden_dim]
        
        # Expression value embeddings
        expr_emb = self.expr_projector(expression.unsqueeze(-1))  # [batch, n_genes, hidden_dim]
        
        # Combine
        combined = torch.cat([gene_emb, expr_emb], dim=-1)  # [batch, n_genes, hidden_dim*2]
        output = self.combine(combined)  # [batch, n_genes, hidden_dim]
        
        return output


print("RawExpressionEncoder defined!")

In [ ]:
class SequenceProjector(nn.Module):
    """Projector for DNA"""
    def __init__(self, input_dim: int, output_dim: int, dropout: float = 0.1):
        super().__init__()
        self.linear = nn.Linear(input_dim, output_dim)
        self.norm = nn.LayerNorm(output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.norm(x)
        x = self.dropout(x)
        return x


class FlashSelfAttentionBlock(nn.Module):
    """Self-Attention with Flash Attention"""
    
    def __init__(self, d_model: int, nhead: int = 8, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.nhead = nhead
        self.head_dim = d_model // nhead
        self.dropout_p = dropout
        
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, return_attention: bool = False):
        batch_size, seq_len, _ = x.shape
        
        Q = self.q_proj(x).view(batch_size, seq_len, self.nhead, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(batch_size, seq_len, self.nhead, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(batch_size, seq_len, self.nhead, self.head_dim).transpose(1, 2)
        
        attn_out = F.scaled_dot_product_attention(
            Q, K, V,
            dropout_p=self.dropout_p if self.training else 0.0,
            is_causal=False
        )
        
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        attn_out = self.out_proj(attn_out)
        
        x = self.norm1(x + self.dropout(attn_out))
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        
        return x, None


class FlashCrossAttentionBlock(nn.Module):
    """Cross-Attention with Flash Attention"""
    
    def __init__(self, d_model: int, nhead: int = 8, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.nhead = nhead
        self.head_dim = d_model // nhead
        self.dropout_p = dropout
        
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query: torch.Tensor, key_value: torch.Tensor):
        batch_size, query_len, _ = query.shape
        key_len = key_value.shape[1]
        
        Q = self.q_proj(query).view(batch_size, query_len, self.nhead, self.head_dim).transpose(1, 2)
        K = self.k_proj(key_value).view(batch_size, key_len, self.nhead, self.head_dim).transpose(1, 2)
        V = self.v_proj(key_value).view(batch_size, key_len, self.nhead, self.head_dim).transpose(1, 2)
        
        attn_out = F.scaled_dot_product_attention(
            Q, K, V,
            dropout_p=self.dropout_p if self.training else 0.0,
            is_causal=False
        )
        
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, query_len, self.d_model)
        attn_out = self.out_proj(attn_out)
        
        x = self.norm1(query + self.dropout(attn_out))
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        
        return x, None


print("Attention blocks defined!")

In [ ]:
class VirtualCellEmbedderDNARNA(nn.Module):
    """
    VCE: DNA + RNA only (NO PROTEIN)
    
    Changes from 3-modality version:
    - Removed protein_query and protein_*_proj
    - Fusion: d_model * 3 -> d_model * 2
    """
    
    def __init__(self, d_model: int, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.nhead = 4
        self.head_dim = d_model // self.nhead
        
        # Queries (2 modalities only)
        self.dna_query = nn.Parameter(torch.randn(1, 1, d_model))
        self.rna_query = nn.Parameter(torch.randn(1, 1, d_model))
        # self.protein_query = nn.Parameter(torch.randn(1, 1, d_model))  # REMOVED
        
        # DNA attention pooling
        self.dna_q_proj = nn.Linear(d_model, d_model)
        self.dna_k_proj = nn.Linear(d_model, d_model)
        self.dna_v_proj = nn.Linear(d_model, d_model)
        self.dna_out_proj = nn.Linear(d_model, d_model)
        
        # RNA attention pooling
        self.rna_q_proj = nn.Linear(d_model, d_model)
        self.rna_k_proj = nn.Linear(d_model, d_model)
        self.rna_v_proj = nn.Linear(d_model, d_model)
        self.rna_out_proj = nn.Linear(d_model, d_model)
        
        # REMOVED: Protein attention pooling
        # self.protein_q_proj = nn.Linear(d_model, d_model)
        # self.protein_k_proj = nn.Linear(d_model, d_model)
        # self.protein_v_proj = nn.Linear(d_model, d_model)
        # self.protein_out_proj = nn.Linear(d_model, d_model)
        
        # Fusion (2 modalities instead of 3)
        self.fusion = nn.Sequential(
            nn.Linear(d_model * 2, d_model * 2),  # Changed from d_model * 3
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model),
            nn.LayerNorm(d_model)
        )

    def _attention_pool(self, query, key_value, q_proj, k_proj, v_proj, out_proj):
        batch_size = key_value.size(0)
        seq_len = key_value.size(1)
        query = query.expand(batch_size, -1, -1)
        
        Q = q_proj(query).view(batch_size, 1, self.nhead, self.head_dim).transpose(1, 2)
        K = k_proj(key_value).view(batch_size, seq_len, self.nhead, self.head_dim).transpose(1, 2)
        V = v_proj(key_value).view(batch_size, seq_len, self.nhead, self.head_dim).transpose(1, 2)
        
        attn_out = F.scaled_dot_product_attention(Q, K, V, is_causal=False)
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, 1, self.d_model)
        attn_out = out_proj(attn_out)
        return attn_out.squeeze(1)

    def forward(self, dna_encoded, rna_encoded):
        """Forward pass with DNA + RNA only (no protein)"""
        dna_pooled = self._attention_pool(
            self.dna_query, dna_encoded,
            self.dna_q_proj, self.dna_k_proj, self.dna_v_proj, self.dna_out_proj
        )
        rna_pooled = self._attention_pool(
            self.rna_query, rna_encoded,
            self.rna_q_proj, self.rna_k_proj, self.rna_v_proj, self.rna_out_proj
        )
        # REMOVED: protein_pooled
        
        # Concat DNA + RNA only (not protein)
        concat = torch.cat([dna_pooled, rna_pooled], dim=-1)  # [batch, d_model*2]
        cell_embedding = self.fusion(concat)
        return cell_embedding


print("VirtualCellEmbedderDNARNA defined (2 modalities only)!")

In [ ]:
class CDTModelDNARNA(nn.Module):
    """
    CDT Model: DNA + RNA only (NO PROTEIN)
    
    Architecture:
    - DNA: Projector -> Self-Attention(2 layers)
    - RNA: RawExpressionEncoder -> Self-Attention(1 layer)
    - Cross-Attention: DNA -> RNA
    - VCE: DNA + RNA pooling
    - Task: Linear -> GELU -> Linear
    
    Removed:
    - protein_projector
    - protein_self_attn_layers
    - rna_to_protein cross-attention
    """

    def __init__(self, config: Optional[CDTDNARNAConfig] = None):
        super().__init__()
        if config is None:
            config = CDTDNARNAConfig()
        self.config = config

        # DNA
        self.dna_projector = SequenceProjector(config.dna_dim, config.hidden_dim, config.dropout)
        self.dna_self_attn_layers = nn.ModuleList([
            FlashSelfAttentionBlock(config.hidden_dim, config.nhead, config.dropout)
            for _ in range(config.dna_self_attn_layers)
        ])

        # RNA (Raw Expression)
        self.rna_encoder = RawExpressionEncoder(config.n_genes, config.hidden_dim, config.dropout)
        self.rna_self_attn_layers = nn.ModuleList([
            FlashSelfAttentionBlock(config.hidden_dim, config.nhead, config.dropout)
            for _ in range(config.rna_self_attn_layers)
        ])

        # REMOVED: Protein
        # self.protein_projector = SequenceProjector(config.protein_dim, config.hidden_dim, config.dropout)
        # self.protein_self_attn_layers = nn.ModuleList([...])

        # Cross-Attention: DNA -> RNA only (no RNA -> Protein)
        self.dna_to_rna = FlashCrossAttentionBlock(config.hidden_dim, config.nhead, config.dropout)
        # REMOVED: self.rna_to_protein

        # VCE (2 modalities)
        self.vce = VirtualCellEmbedderDNARNA(config.hidden_dim, config.dropout)

        # Task
        self.task_layer = nn.Sequential(
            nn.Linear(config.hidden_dim, config.hidden_dim),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.hidden_dim, config.n_genes)
        )

    def forward(self, dna_emb, rna_expr):
        """
        Forward pass with DNA + RNA only (no protein_emb argument)
        
        Args:
            dna_emb: [batch, 896, 3072] DNA embeddings
            rna_expr: [batch, n_genes] raw expression values
        
        Returns:
            logits: [batch, n_genes] predicted expression
        """
        batch_size = dna_emb.size(0)

        # Encode DNA
        dna = self.dna_projector(dna_emb)
        
        # Encode RNA
        rna = self.rna_encoder(rna_expr)
        
        # REMOVED: Encode Protein
        # protein = self.protein_projector(protein_emb)
        # protein = protein.unsqueeze(0).expand(batch_size, -1, -1)

        # Self-Attention: DNA
        for layer in self.dna_self_attn_layers:
            dna, _ = layer(dna)

        # Self-Attention: RNA
        for layer in self.rna_self_attn_layers:
            rna, _ = layer(rna)

        # REMOVED: Self-Attention: Protein
        # for layer in self.protein_self_attn_layers:
        #     protein, _ = layer(protein)

        # Cross-Attention: DNA -> RNA
        rna, _ = self.dna_to_rna(query=rna, key_value=dna)
        
        # REMOVED: Cross-Attention: RNA -> Protein
        # protein, _ = self.rna_to_protein(query=protein, key_value=rna)

        # VCE (DNA + RNA only)
        cell_embedding = self.vce(dna, rna)
        
        # Task
        logits = self.task_layer(cell_embedding)

        return logits

    def get_num_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


print("CDTModelDNARNA defined (DNA + RNA only)!")

## 5. Dataset

In [ ]:
class CDTDNARNADataset(Dataset):
    """
    CDT Dataset: DNA + RNA only (no Protein)
    
    Same as CDTRawExpressionDatasetV2 but without protein references.
    """
    
    def __init__(self, data_path, 
                 shared_rna_expr, shared_rna_coord_to_idx,
                 shared_dna_emb, shared_dna_coord_to_idx,
                 symbol_to_idx, ensg_to_symbol):
        
        self.data_path = Path(data_path)
        self.symbol_to_idx = symbol_to_idx
        self.ensg_to_symbol = ensg_to_symbol
        
        # Load raw data
        with h5py.File(data_path, 'r') as f:
            raw_betas = f['beta'][:]
            raw_enh_chr = [c.decode() if isinstance(c, bytes) else c for c in f['enhancer_chr'][:]]
            raw_enh_start = f['enhancer_start'][:]
            raw_enh_end = f['enhancer_end'][:]
            raw_gene_ids = [g.decode() if isinstance(g, bytes) else g for g in f['gene_ids'][:]]
        
        raw_enh_centers = (raw_enh_start + raw_enh_end) // 2
        
        # Filter samples
        print(f"\n[Filtering] {self.data_path.name}")
        print(f"  Original samples: {len(raw_betas)}")
        
        valid_indices = []
        gene_indices = []
        
        for i, ensg in enumerate(raw_gene_ids):
            symbol = ensg_to_symbol.get(ensg, None)
            if symbol is not None and symbol in symbol_to_idx:
                valid_indices.append(i)
                gene_indices.append(symbol_to_idx[symbol])
        
        n_filtered = len(raw_betas) - len(valid_indices)
        print(f"  Filtered out: {n_filtered} samples")
        print(f"  Remaining: {len(valid_indices)} samples")
        
        # Apply filtering
        self.betas = raw_betas[valid_indices]
        self.enh_chr = [raw_enh_chr[i] for i in valid_indices]
        self.enh_centers = raw_enh_centers[valid_indices]
        self.gene_indices = gene_indices
        
        # Store shared data
        self.dna_emb = shared_dna_emb
        self.dna_coord_to_idx = shared_dna_coord_to_idx
        self.rna_expr = shared_rna_expr
        self.rna_coord_to_idx = shared_rna_coord_to_idx
        
        # Check RNA coordinate matching
        rna_matched = sum(1 for i in range(len(self.enh_chr)) 
                     if (self.enh_chr[i], self.enh_centers[i]) in self.rna_coord_to_idx)
        
        print(f"  RNA coord matched: {rna_matched}/{len(self)} ({100*rna_matched/len(self):.1f}%)")
        print(f"  Beta: mean={np.mean(self.betas):.4f}, std={np.std(self.betas):.4f}")
    
    def __len__(self):
        return len(self.betas)
    
    def __getitem__(self, idx):
        coord = (self.enh_chr[idx], self.enh_centers[idx])
        
        # DNA embedding
        dna_idx = self.dna_coord_to_idx.get(coord, 0)
        dna = torch.from_numpy(self.dna_emb[dna_idx]).float()
        
        # RNA expression (raw)
        rna_idx = self.rna_coord_to_idx.get(coord, 0)
        rna_expr = torch.from_numpy(self.rna_expr[rna_idx]).float()
        
        gene_idx = torch.tensor(self.gene_indices[idx], dtype=torch.long)
        beta = torch.tensor(self.betas[idx], dtype=torch.float32)
        
        return dna, rna_expr, gene_idx, beta


print("CDTDNARNADataset defined!")

## 6. Load DNA Embeddings (No Protein)

In [ ]:
print("="*60)
print("Loading DNA embeddings (NO Protein - Protein-free version!)")
print("="*60)

# DNA embeddings
print("\n[1/1] Loading DNA embeddings...")
with h5py.File(DNA_PATH, 'r') as f:
    SHARED_DNA_EMB = f['embeddings'][:]
    dna_centers = f['centers'][:]
    dna_chroms = [c.decode() if isinstance(c, bytes) else c for c in f['chroms'][:]]

SHARED_DNA_COORD_TO_IDX = {}
for i, (chrom, center) in enumerate(zip(dna_chroms, dna_centers)):
    SHARED_DNA_COORD_TO_IDX[(chrom, center)] = i

print(f"  Shape: {SHARED_DNA_EMB.shape}")

# NO PROTEIN LOADING!
print("\n[SKIPPED] Protein embeddings (Protein-free version)")

print("\n" + "="*60)
print("DNA embeddings loaded! (Protein-free)")
print("="*60)

## 7. Training Functions (No Protein)

In [ ]:
def train_epoch(model, dataloader, optimizer, criterion, device):
    """Training function without protein_emb argument"""
    model.train()
    total_loss = 0
    all_preds = []
    all_targets = []

    for dna, rna_expr, gene_idx, beta in tqdm(dataloader, desc="Training"):
        dna = dna.to(device)
        rna_expr = rna_expr.to(device)
        gene_idx = gene_idx.to(device)
        beta = beta.to(device)

        optimizer.zero_grad()

        # NO protein_emb argument
        logits = model(dna, rna_expr)
        
        batch_size = logits.shape[0]
        pred = logits[range(batch_size), gene_idx]
        
        loss = criterion(pred, beta)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        all_preds.extend(pred.detach().cpu().numpy())
        all_targets.extend(beta.cpu().numpy())

    from scipy.stats import pearsonr
    train_r, _ = pearsonr(all_preds, all_targets)

    return total_loss / len(dataloader), train_r


def evaluate(model, dataloader, criterion, device):
    """Evaluation function without protein_emb argument"""
    model.eval()
    total_loss = 0
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for dna, rna_expr, gene_idx, beta in tqdm(dataloader, desc="Evaluating"):
            dna = dna.to(device)
            rna_expr = rna_expr.to(device)
            gene_idx = gene_idx.to(device)
            beta = beta.to(device)
            
            # NO protein_emb argument
            logits = model(dna, rna_expr)
            
            batch_size = logits.shape[0]
            pred = logits[range(batch_size), gene_idx]
            
            loss = criterion(pred, beta)
            
            total_loss += loss.item()
            all_preds.extend(pred.cpu().numpy())
            all_targets.extend(beta.cpu().numpy())
    
    from scipy.stats import pearsonr
    r, p = pearsonr(all_preds, all_targets)
    
    return total_loss / len(dataloader), r


print("Training functions defined (no protein_emb)!")

## 8. Configuration

In [ ]:
# Hyperparameters
config = {
    'hidden_dim': 768,
    'nhead': 8,
    'dropout': 0.3,
    
    'batch_size': 8,
    'learning_rate': 1e-4,
    'weight_decay': 1e-5,
    'epochs': 50,
    'patience': 10,
    'early_stop_min_delta': 0.001,
    'huber_delta': 1.0,
    
    'n_genes': 2360,
    'max_cells_per_enhancer': MAX_CELLS_PER_ENHANCER
}

print("Config (DNA + RNA Only - NO PROTEIN):")
for k, v in config.items():
    print(f"  {k}: {v}")

## 9. Create Datasets

In [ ]:
print("Creating datasets (DNA + RNA only)...")
print("="*60)

# RNA coord to idx mapping
SHARED_RNA_COORD_TO_IDX = {coord: i for i, coord in enumerate(unique_coords)}

train_dataset = CDTDNARNADataset(
    TRAIN_PATH, 
    SHARED_RNA_EXPR,
    SHARED_RNA_COORD_TO_IDX,
    SHARED_DNA_EMB, 
    SHARED_DNA_COORD_TO_IDX,
    SYMBOL_TO_IDX,
    ENSG_TO_SYMBOL
)

print("\n" + "-"*60)

val_dataset = CDTDNARNADataset(
    VAL_PATH, 
    SHARED_RNA_EXPR,
    SHARED_RNA_COORD_TO_IDX,
    SHARED_DNA_EMB, 
    SHARED_DNA_COORD_TO_IDX,
    SYMBOL_TO_IDX,
    ENSG_TO_SYMBOL
)

train_loader = DataLoader(
    train_dataset, 
    batch_size=config['batch_size'], 
    shuffle=True,
    drop_last=True,
    num_workers=0
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=config['batch_size'], 
    shuffle=False,
    num_workers=0
)

print("\n" + "="*60)
print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print("="*60)

## 10. Create Model

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# NO protein_emb_shared!
# protein_emb_shared = torch.from_numpy(...).to(device)  # REMOVED

model_config = CDTDNARNAConfig(
    dna_dim=3072,
    dna_seq_len=896,
    # protein_dim=768,  # REMOVED
    n_genes=config['n_genes'],
    hidden_dim=config['hidden_dim'],
    nhead=config['nhead'],
    dropout=config['dropout']
)

model = CDTModelDNARNA(model_config).to(device)

total_params = model.get_num_params()
print(f"\nModel: CDTModelDNARNA (DNA + RNA only, NO PROTEIN)")
print(f"  DNA: Projector -> Self-Attn(2 layers)")
print(f"  RNA: RawExpressionEncoder -> Self-Attn(1 layer)")
print(f"  Cross-Attn: DNA -> RNA")
print(f"  VCE: DNA + RNA pooling")
print(f"  hidden_dim: {config['hidden_dim']}")
print(f"  n_genes: {config['n_genes']}")
print(f"  Total parameters: {total_params:,}")

## 11. Training

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(), 
    lr=config['learning_rate'],
    weight_decay=config['weight_decay']
)

criterion = nn.SmoothL1Loss(beta=config['huber_delta'])

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5
)

print(f"Optimizer: AdamW (lr={config['learning_rate']}, weight_decay={config['weight_decay']})")
print(f"Loss: Huber (SmoothL1Loss, delta={config['huber_delta']})")
print(f"Scheduler: ReduceLROnPlateau (factor=0.5, patience=5)")

best_r = -1
patience_counter = 0
history = {'train_loss': [], 'train_r': [], 'val_loss': [], 'val_r': []}
min_delta = config['early_stop_min_delta']

print(f"EarlyStopping: patience={config['patience']}, min_delta={min_delta}")
print(f"\nStarting training for {config['epochs']} epochs...")
print("="*60)

for epoch in range(config['epochs']):
    # NO protein_emb_shared argument!
    train_loss, train_r = train_epoch(
        model, train_loader, optimizer, criterion, device
    )
    
    val_loss, val_r = evaluate(
        model, val_loader, criterion, device
    )
    
    history['train_loss'].append(train_loss)
    history['train_r'].append(train_r)
    history['val_loss'].append(val_loss)
    history['val_r'].append(val_r)
    
    scheduler.step(val_r)
    current_lr = optimizer.param_groups[0]['lr']
    
    print(f"Epoch {epoch+1}/{config['epochs']}: "
          f"Train Loss={train_loss:.4f}, Train r={train_r:.4f}, "
          f"Val Loss={val_loss:.4f}, Val r={val_r:.4f} (lr={current_lr:.2e})")
    
    if val_r > best_r + min_delta:
        best_r = val_r
        patience_counter = 0
        torch.save(model.state_dict(), OUTPUT_BASE / "cdt_v2_dna_rna_best.pt")
        print(f"  -> New best model saved! (r={best_r:.4f})")
    else:
        patience_counter += 1
    
    if patience_counter >= config['patience']:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

print("="*60)
print(f"Training complete! Best r={best_r:.4f}")
print(f"\nComparison:")
print(f"  CDT v3.3.1 (scGPT + ProteomeLM): r = 0.50")
print(f"  CDT Raw v2 (Raw + ProteomeLM): r = 0.5034")
print(f"  CDT v2 DNA+RNA (Raw, NO Protein): r = {best_r:.4f}")

## 12. Results

In [ ]:
import matplotlib.pyplot as plt

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Val')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (Huber)')
axes[0].legend()
axes[0].set_title('Training Loss')

axes[1].plot(history['train_r'], label='Train r')
axes[1].plot(history['val_r'], label='Val r')
axes[1].axhline(y=0.50, color='r', linestyle='--', label='v3.3.1 (scGPT+Protein): 0.50')
axes[1].axhline(y=0.5034, color='g', linestyle='--', label='Raw v2 (Raw+Protein): 0.5034')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Pearson r')
axes[1].legend()
axes[1].set_title('CDT v2: DNA + RNA Only (NO Protein)')

plt.tight_layout()
plt.savefig(OUTPUT_BASE / f"training_history_dna_rna_{timestamp}.png", dpi=150)
plt.show()

# Convert numpy types to Python native types for JSON serialization
def convert_to_serializable(obj):
    """Convert numpy types to Python native types for JSON serialization."""
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, (np.float32, np.float64)):
        return float(obj)
    elif isinstance(obj, (np.int32, np.int64)):
        return int(obj)
    elif isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_serializable(v) for v in obj]
    return obj

results = {
    'version': 'v2_dna_rna_only',
    'description': 'CDT v2: DNA + RNA Only (NO Protein)',
    'architecture': {
        'dna': 'Enformer embeddings',
        'rna': 'Raw expression (RawExpressionEncoder)',
        'protein': 'REMOVED (Protein-free)',
        'cross_attention': 'DNA -> RNA only',
        'vce': '2 modalities (DNA + RNA)'
    },
    'comparison': {
        'v3_3_1_scgpt_protein': 0.50,
        'raw_v2_with_protein': 0.5034,
        'v2_dna_rna_only': float(best_r)
    },
    'config': config,
    'best_r': float(best_r),
    'history': convert_to_serializable(history),
    'timestamp': timestamp,
    'train_samples': len(train_dataset),
    'val_samples': len(val_dataset),
}

with open(OUTPUT_BASE / f"results_dna_rna_{timestamp}.json", 'w') as f:
    json.dump(results, f, indent=2)

print(f"\nResults saved to {OUTPUT_BASE}")
print(f"\n" + "="*60)
print("CDT v2: DNA + RNA Only Results:")
print(f"  v3.3.1 (scGPT + ProteomeLM): r = 0.50")
print(f"  Raw v2 (Raw + ProteomeLM): r = 0.5034")
print(f"  v2 DNA+RNA (NO Protein): r = {best_r:.4f}")
print("="*60)

if best_r >= 0.45:
    print("\n*** SUCCESS: Protein-free version achieves practical performance! ***")
    print("-> Proteomics data is NOT required for this task")
elif best_r >= 0.40:
    print("\nResult: Acceptable performance without Protein")
    print("-> Protein adds some value but not essential")
else:
    print("\nResult: Protein seems important")
    print("-> Consider keeping Protein branch for best results")

## Done!

### Summary

| Version | DNA | RNA | Protein | Val r |
|---------|-----|-----|---------|-------|
| v3.3.1 | Enformer | scGPT | ProteomeLM | 0.50 |
| Raw v2 | Enformer | Raw | ProteomeLM | 0.5034 |
| **v2 DNA+RNA** | Enformer | Raw | **None** | ??? |

### Advantages of Protein-free version
1. **Accessible**: No Proteomics data required
2. **Simple**: Fewer modalities to manage
3. **Lightweight**: ~30% fewer parameters
4. **Fast**: Faster training and inference

### Attention Maps preserved
- ✅ DNA Self-Attention: [batch, 896, 896]
- ✅ RNA Self-Attention: [batch, n_genes, n_genes]
- ✅ DNA → RNA Cross-Attention: [batch, n_genes, 896]
- ❌ RNA → Protein Cross-Attention: Removed